In [5]:
from transformers import AutoTokenizer
import json
from datasets import Dataset
import numpy as np
# from skmultilearn.model_selection import iterative_train_test_split
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

In [6]:
file_path1 = "data/stratified_test_data_pinfo.json"  # Update with your actual file path
with open(file_path1, "r") as f:
    data1 = json.load(f)
file_path2 = "data/stratified_train_data_pinfo.json"  # Update with your actual file path
with open(file_path2, "r") as f:
    data2 = json.load(f)

In [7]:
len(data1 + data2)

837

In [8]:
len(data1)

192

In [9]:
data1[:2]

[{'text': "From Patient: Dr.Person1 I need my prescription sent to the pharmacy for my flecainide acetate 100 mg tablets twice a day the pharmacist has try requesting it no success and I don't have any pills. Person2",
  'labels': ['CareCoordinationPatient_None',
   'PartnershipPatient_Clinical care',
   'PartnershipPatient_activeParticipation/involvement',
   'SDOH_HealthCareAccessAndQuality',
   'SDOH_NeighborhoodAndBuiltEnvironment']},
 {'text': 'From Patient: Hi Dr. Person1,I left two messages last week on MM/DD/YYYY and MM/DD/YYYY. Asking about stopping the eliquis for the second time on this month since I had the first procedure of diagnostic lumbar facet block.RF ABLATION.AND DR.Person 2 wants to know for my second dose is scheduled for MM/DD/YYYY he wants to know if I cud have it done that date or cud schedule it a bit further date,since I have to stop the eliquis three days prior to the procedure. Thank you, Person2',
  'labels': ['CareCoordinationPatient_None',
   'Partnershi

In [19]:
from collections import defaultdict
# get the dist
def get_dist(data):
    code_dist = defaultdict(int)
    subcode_dist = defaultdict(int)
    combo_dist = defaultdict(int)
    for item in data:
        for l in item['labels']:
            code, subcode = l.split("_")
            if subcode == "Clinical care":
                subcode = "Clinical Care"
            code_dist[code] += 1
            subcode_dist[subcode] += 1
            combo_dist[l] += 1
    return code_dist, subcode_dist, combo_dist
# get the dist
def get_dist_message_wise(data):
    # remove dups in message
    code_dist = defaultdict(int)
    subcode_dist = defaultdict(int)
    combo_dist = defaultdict(int)
    for item in data:
        code_set = set()
        subcode_set = set()
        combo_set = set()
        for l in item['labels']:
            code, subcode = l.split("_")
            if subcode == "Clinical care":
                subcode = "Clinical Care"
            code_set.add(code)
            subcode_set.add(subcode)
            combo_set.add(l)
        for code in code_set:
            code_dist[code] += 1
        for subcode in subcode_set:
            subcode_dist[subcode] += 1
        for combo in combo_set:
            combo_dist[combo] += 1
    return code_dist, subcode_dist, combo_dist
cd, sd, cmd = get_dist_message_wise(data1)

In [20]:
cd

defaultdict(int,
            {'CareCoordinationPatient': 33,
             'PartnershipPatient': 113,
             'SDOH': 98,
             'SharedDecisionPatient': 24,
             'SharedDecisionProvider': 17,
             'PartnershipProvider': 60,
             'SocioEmotionalBehaviour': 45,
             'CareCoordinationProvider': 26})

In [23]:
count = 0
for k,v in cd.items():
    count += v
count
    

416

In [21]:
len(data1)

192

In [22]:
len(data2)

645